# 🍅 Tomato Market Price Trend Analysis
## Maharashtra APMC Markets | Jan 2025 – Aug 2026

**Author:** Aasif  
**Date:** August 2026  
**Data Source:** Agmarknet — Government of India  

---

### About This Notebook

This is the first notebook in the analysis pipeline. The goal here is to:

1. Load the raw dataset and verify it loaded correctly
2. Understand what every column means
3. Check data quality — nulls, duplicates, outliers
4. Get a first feel for the numbers before any visualisation

> **Why do this before charts?**  
> Because charts lie if your data has problems you haven't caught yet.  
> This notebook is the foundation everything else builds on.

---

**Markets Covered:**
| Market | District | Role |
|--------|----------|------|
| Pimpalgaon Baswant APMC | Nashik | Primary Production Hub |
| Nasik APMC | Nashik | Secondary Production Hub |
| Mumbai APMC | Mumbai | Major Consumption Market |
| Pune APMC | Pune | Major Consumption Market |
| Pune (Manjri) APMC | Pune | Secondary Consumption |
| Nagpur APMC | Nagpur | Inland Distribution Hub |

---
## Section 1 — Imports and Setup

We load only what we need for exploration.  
No matplotlib yet — that comes in the visualisation notebook.  
Keeping imports minimal = faster load + cleaner code.

In [ ]:
import pandas as pd
import numpy as np

# Display settings — makes output readable in notebook
pd.set_option('display.max_columns', None)      # show all columns
pd.set_option('display.float_format', '{:.2f}'.format)  # 2 decimal places

print("Libraries loaded ✅")
print(f"Pandas version  : {pd.__version__}")
print(f"NumPy version   : {np.__version__}")

---
## Section 2 — Loading the Data

We load the CSV from the `data/` folder using a relative path.  
Using `../data/` because our notebook is inside `notebooks/` — 
so we go one level up (`..`) then into `data/`.

> **Golden Rule:** Raw data lives in `data/` and is **never modified.**  
> Every transformation happens in code, not in the file itself.  
> This means anyone can re-run this notebook and get the same result.

In [ ]:
# Load the raw dataset
df = pd.read_csv('../data/master_df.csv')

# First confirmation — did it load?
print(f"✅ Dataset loaded")
print(f"   Rows      : {df.shape[0]:,}")
print(f"   Columns   : {df.shape[1]}")
print(f"\nColumn names:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i}. {col}")

---
## Section 3 — Understanding the Columns

Before we touch any numbers, we need to know what each column *means*  
in the real world. This is called **domain understanding** —  
it's what separates an analyst from someone who just runs code.

In [ ]:
# Look at the first 5 rows — our "first handshake" with the data
print("=== FIRST 5 ROWS ===")
print(df.head(5).to_string())

print("\n=== COLUMN DATA TYPES ===")
print(df.dtypes)

### What Each Column Means

| Column | Type | What It Means |
|--------|------|---------------|
| `state` | text | Always "Maharashtra" — single state dataset |
| `district` | text | The district the market belongs to |
| `market` | text | The specific APMC market name |
| `commodity_group` | text | Always "Vegetables" — single category |
| `commodity` | text | Always "Tomato" — single commodity |
| `date` | text* | The trading date — *needs to be converted to datetime* |
| `arrival_quantity` | number | How many Metric Tonnes arrived at market that day |
| `arrival_unit` | text | Always "Metric Tonnes" — unit of arrival |
| `modal_price` | number | The most common transaction price that day (Rs./Quintal) |
| `price_unit` | text | Always "Rs./Quintal" — unit of price |

> **Note:** `date` is stored as text (`object`) — we will convert it  
> to a proper datetime type in the next step. This is very common  
> with CSV files. Dates almost always come in as text.

> **Note:** `modal_price` is in **Rs. per Quintal** (1 Quintal = 100 kg).  
> So ₹1,400/quintal = ₹14/kg at wholesale level.

In [ ]:
# Convert date column from text to actual datetime
df['date'] = pd.to_datetime(df['date'])

# Derive time-based columns
df['year'] = df['date'].dt.year
df['month_num'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()
df['quarter'] = df['date'].dt.quarter
df['week'] = df['date'].dt.isocalendar().week

# Add short market names for cleaner chart labels
market_short_map = {
    'Mumbai APMC': 'Mumbai',
    'Nagpur APMC': 'Nagpur',
    'Nasik APMC': 'Nasik',
    'Pimpalgaon Baswant APMC': 'Pimpalgaon',
    'Pune APMC': 'Pune',
    'Pune(Manjri) APMC': 'Pune Manjri'
}
df['market_short'] = df['market'].map(market_short_map)

# Quick check — confirm the new columns look right
print(df.dtypes)
print()
df[['date', 'year', 'month_num', 'month_name', 'quarter', 'week', 'market', 'market_short']].head(5)

In [ ]:
df['date'] = pd.to_datetime(df['date'])

df['year'] = df['date'].dt.year
df['month_num'] = df['date'].dt.month
df['month_name'] = df['date'].dt.month_name()
df['quarter'] = df['date'].dt.quarter
df['week'] = df['date'].dt.isocalendar().week

market_short_map = {
    'Mumbai Apmc': 'Mumbai',
    'Nagpur Apmc': 'Nagpur',
    'Nasik Apmc': 'Nasik',
    'Pimpalgaon Baswant Apmc': 'Pimpalgaon',
    'Pune Apmc': 'Pune',
    'Pune(Manjri) Apmc': 'Pune Manjri'
}
df['market_short'] = df['market'].map(market_short_map)

print(df.dtypes)
df[['date','year','month_num','month_name','quarter','week','market','market_short']].head()

In [ ]:
df.describe()

## Section 4 — Date Conversion + Derived Columns
Converting date to datetime and extracting useful time features.

In [ ]:
# Convert date from text → datetime
df['date'] = pd.to_datetime(df['date'])

# Derive time columns — we'll use these heavily in analysis
df['year']       = df['date'].dt.year
df['month_num']  = df['date'].dt.month
df['month_name'] = df['date'].dt.strftime('%B')
df['quarter']    = df['date'].dt.quarter
df['week']       = df['date'].dt.isocalendar().week.astype(int)

# Short market names — for clean readable output
SHORT = {
    'Mumbai Apmc'            : 'Mumbai',
    'Nagpur Apmc'            : 'Nagpur',
    'Nasik Apmc'             : 'Nasik',
    'Pimpalgaon Baswant Apmc': 'Pimpalgaon',
    'Pune Apmc'              : 'Pune',
    'Pune(Manjri) Apmc'      : 'Pune Manjri',
}
df['market_short'] = df['market'].map(SHORT)

# Confirm
print("Date dtype    :", df['date'].dtype)
print("Date range    :", df['date'].min().date(), "→", df['date'].max().date())
print("New columns   :", ['year','month_num','month_name','quarter','week','market_short'])
print("\nSample row:")
print(df[['date','year','month_num','quarter','market_short','modal_price']].head(3).to_string())

## Section 5 — Data Quality Check
Nulls, duplicates, zero arrivals, coverage per market.

In [ ]:
print("=== NULL CHECK ===")
print(df.isnull().sum())

print("\n=== DUPLICATES ===")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\n=== ZERO ARRIVAL ROWS ===")
zero = df[df['arrival_quantity'] == 0]
print(f"Count: {len(zero)}")
print(zero[['market_short','date','arrival_quantity','modal_price']].to_string())

print("\n=== RECORDS PER MARKET ===")
print(df['market_short'].value_counts().to_string())

print("\n=== DATE COVERAGE PER MARKET ===")
for mkt in df['market_short'].unique():
    sub = df[df['market_short'] == mkt]
    print(f"{mkt:15s}: {sub['date'].min().date()} → {sub['date'].max().date()} ({len(sub)} records)")

## Section 5 — Numeric Statistics
The raw numbers. We look at both columns that actually vary — price and arrival quantity.

In [ ]:
print("=== PRICE + ARRIVAL STATISTICS ===\n")
print(df[['arrival_quantity','modal_price']].describe(
    percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
).round(2).to_string())

print("\n=== PER MARKET PRICE SUMMARY ===\n")
mkt_stats = df.groupby('market_short')['modal_price'].agg(
    Min='min', Max='max', Mean='mean', Median='median',
    Std='std', CV=lambda x: round((x.std()/x.mean())*100, 1)
).round(1).sort_values('Mean', ascending=False)
print(mkt_stats.to_string())

print("\n=== PER MARKET ARRIVAL SUMMARY (MT) ===\n")
arr_stats = df.groupby('market_short')['arrival_quantity'].agg(
    Total='sum', Daily_Avg='mean', Max_Day='max'
).round(1).sort_values('Total', ascending=False)
print(arr_stats.to_string())

### What the Numbers Tell Us

- **Price range** is ₹200 → ₹10,500 — a 52x gap. Tomatoes are one of India's 
  most volatile commodities.
- **Mean (₹1,677) > Median (₹1,300)** — the distribution is right-skewed. 
  A few extreme high-price days are pulling the average up.
- **95th percentile is ₹3,900** — meaning 95% of all trading days had prices 
  below ₹3,900. Anything above is a true outlier event.
- **Nagpur** has the highest average price (₹2,188) and highest volatility 
  (CV ~62%) — it's far from supply, so any disruption hits it hard.
- **Nasik** is cheapest (₹1,084 avg) — production is right there.
- **Pimpalgaon** handles ~49% of all volume — it's the real engine of 
  Maharashtra's tomato supply chain.

In [ ]:
df.columns

In [ ]:
print("=== MONTHLY AVG PRICE — ALL MARKETS ===\n")
monthly = df.groupby('month_name').agg(
    Avg_Price=('modal_price','mean'),
    Total_Arrivals=('arrival_quantity','sum'),
    Records=('modal_price','count')
).round(1)
print(monthly.to_string())

print("\n=== YEAR SUMMARY ===\n")
print(df.groupby('year')['modal_price'].agg(
    Mean='mean', Median='median', Min='min', Max='max'
).round(1).to_string())

---
## Summary — What We Know Before Charting

| Finding | Value |
|---------|-------|
| Total records | 3,093 |
| Zero nulls | ✅ Clean data |
| Price range | ₹200 – ₹10,500/quintal |
| Most expensive market | Nagpur (avg ₹2,188) |
| Cheapest market | Nasik (avg ₹1,084) |
| Highest volume market | Pimpalgaon (~49% of all supply) |
| Peak price months | Jul–Aug, Nov–Dec |
| Low price months | Jan–Apr (Rabi harvest) |
| Most volatile | Nagpur (CV 62%) |
| Most stable | Pune Manjri (CV ~46%) |

